## About Dataset
Context
This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content
5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset.
Columns

- asin - ID of the product, like B000FA64PK
- helpful - helpfulness rating of the review - example: 2/3.
- overall - rating of the product.
- reviewText - text of the review (heading).
- reviewTime - time of the review (raw).
- reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
- reviewerName - name of the reviewer.
- summary - summary of the review (description).
- unixReviewTime - unix timestamp.

Acknowledgements
This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

Inspiration
- Sentiment analysis on reviews.
- Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
- Fake reviews/ outliers.
- Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
- Any other interesting analysis

In [1]:
import pandas as pd
import numpy as np
import nltk
import re

In [2]:
#Loading dataset
data=pd.read_csv('data/all_kindle_review.csv',index_col=0)

In [3]:
#First view of data
data.head()

,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [4]:
#Relivent columns
data=data[['reviewText','rating']]

In [5]:
data.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [6]:
data.shape

(12000, 2)

In [7]:
#checking missing values
data.isna().sum()

reviewText    0
rating        0
dtype: int64

In [8]:
#Unique value in ratings
data['rating'].unique()

array([3, 5, 4, 2, 1])

In [9]:
#checking for imbalance dataset
data['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

### Preprocessing and cleaning

In [10]:
#positive review is 1 and negative review is 0
data['rating']=data['rating'].apply(lambda x: 0 if x<3 else 1)

In [11]:
data['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [12]:
#Lowering the review text
data['reviewText']=data['reviewText'].str.lower()

In [13]:
from nltk.corpus import stopwords
nltk.download('stopwords')
from bs4 import BeautifulSoup

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\himan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
# special characters
data['reviewText'] = data['reviewText'].apply(
    lambda x: re.sub(r'[^a-zA-Z0-9]+', ' ', str(x))
)

# stopwords
stop_words = set(stopwords.words('english'))

data['reviewText'] = data['reviewText'].apply(
    lambda x: " ".join(
        word for word in x.split()
        if word.lower() not in stop_words
    )
)

# URL
data['reviewText'] = data['reviewText'].apply(
    lambda x: re.sub(
        r'https?://\S+|www\.\S+',
        '',
        str(x)
    )
)

# HTML tags
data['reviewText'] = data['reviewText'].apply(
    lambda x: BeautifulSoup(str(x), 'html.parser').get_text()
)

# additional spaces
data['reviewText'] = data['reviewText'].apply(
    lambda x: " ".join(x.split())
)

In [17]:
data.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [19]:
#Lemmatizer
from nltk.stem import WordNetLemmatizer

In [20]:
lemma=WordNetLemmatizer()

In [23]:
#lemmatizing the text
def lemma_word(text):
    return " ".join([ lemma.lemmatize(word) for word in text.split()])

In [24]:
data['reviewText'] = data['reviewText'].apply(
    lambda x: lemma_word(x))

In [25]:
data['reviewText'].head()

0    jace rankin may short he nothing mess man haul...
1    great short read didnt want put read one sitti...
2    ill start saying first four book wasnt expecti...
3    aggie angela lansbury carry pocketbook instead...
4    expect type book library pleased find price right
Name: reviewText, dtype: str

### Train test split

In [27]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(data['reviewText'],data['rating'],test_size=0.20)

### BOW,TF-IDF,WORD2VEC IMPLEMENTATION

In [41]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
x_train_bow=bow.fit_transform(x_train).toarray()
x_test_bow=bow.transform(x_test).toarray()

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfi=TfidfVectorizer()
x_train_tf=tfi.fit_transform(x_train).toarray()
x_test_tf=tfi.transform(x_test).toarray()

### ML ALGORITHM 

In [47]:
from sklearn.naive_bayes import GaussianNB

In [48]:
nb_model_bow=GaussianNB().fit(x_train_bow,y_train)
nb_model_tfi=GaussianNB().fit(x_train_tf,y_train)

### Metrics check

In [39]:
from sklearn.metrics import accuracy_score,classification_report

In [49]:
y_pred_bow=nb_model_bow.predict(x_test_bow)
y_pred_tf=nb_model_tfi.predict(x_test_tf)

In [56]:
print("TFIDF Accuracy:",accuracy_score(y_pred_tf,y_test))

TFIDF Accuracy: 0.6004166666666667


In [57]:
print("Bow accuracy :",accuracy_score(y_pred_bow,y_test))

Bow accuracy : 0.5958333333333333


In [58]:
print("Report tf:",classification_report(y_pred_tf,y_test))

Report tf:               precision    recall  f1-score   support

           0       0.66      0.44      0.53      1218
           1       0.57      0.77      0.65      1182

    accuracy                           0.60      2400
   macro avg       0.61      0.60      0.59      2400
weighted avg       0.62      0.60      0.59      2400



In [59]:
print("Report bow:",classification_report(y_pred_bow,y_test))

Report bow:               precision    recall  f1-score   support

           0       0.66      0.44      0.53      1233
           1       0.56      0.77      0.65      1167

    accuracy                           0.60      2400
   macro avg       0.61      0.60      0.59      2400
weighted avg       0.61      0.60      0.59      2400

